In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Normal Nuc seq data 9 sample (7+2 add data)
Path10x_4_1='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-1_GEX/outs/filtered_feature_bc_matrix'
adata_norm_01 = sc.read_10x_mtx(Path10x_4_1,var_names='gene_symbols',cache=True)
Path10x_4_3='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-3_GEX/outs/filtered_feature_bc_matrix'
adata_norm_02 = sc.read_10x_mtx(Path10x_4_3,var_names='gene_symbols',cache=True)
Path10x_4_8='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-8_GEX/outs/filtered_feature_bc_matrix'
adata_norm_03 = sc.read_10x_mtx(Path10x_4_8,var_names='gene_symbols',cache=True)
Path10x_4_9='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-9_GEX/outs/filtered_feature_bc_matrix'
adata_norm_04 = sc.read_10x_mtx(Path10x_4_9,var_names='gene_symbols',cache=True)
Path10x_4_13='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-13_GEX/outs/filtered_feature_bc_matrix'
adata_norm_05 = sc.read_10x_mtx(Path10x_4_13,var_names='gene_symbols',cache=True)
Path10x_4_88='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-88_GEX/outs/filtered_feature_bc_matrix'
adata_norm_06 = sc.read_10x_mtx(Path10x_4_88,var_names='gene_symbols',cache=True)
Path10x_4_89='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-89_GEX/outs/filtered_feature_bc_matrix'
adata_norm_07 = sc.read_10x_mtx(Path10x_4_89,var_names='gene_symbols',cache=True)
Path10x_4_8_add='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-8_GEX/outs/filtered_feature_bc_matrix'
adata_norm_08 = sc.read_10x_mtx(Path10x_4_8_add,var_names='gene_symbols',cache=True)
Path10x_4_88_add='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02a_count_add/4-88_GEX/outs/filtered_feature_bc_matrix'
adata_norm_09 = sc.read_10x_mtx(Path10x_4_88_add,var_names='gene_symbols',cache=True)

adatas = [adata_norm_01,adata_norm_02,adata_norm_03,adata_norm_04,adata_norm_05\
          ,adata_norm_06,adata_norm_07,adata_norm_08, adata_norm_09]

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatas):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
for i, adata in enumerate(adatas):
    plt.figure(figsize=(8, 6))
    
    # 산점도 그리기 (c 파라미터에 mt_frac 사용, vmin/vmax 지정)
    scatter = plt.scatter(
        adata.obs['n_counts'], 
        adata.obs['n_genes'], 
        c=adata.obs['mt_frac'],  # 점의 색을 mt_frac으로 지정
        cmap='viridis',          # 컬러맵 선택
        alpha=0.7,
        vmin=0, vmax=1          
    )
    
    # 컬러바 추가
    plt.colorbar(scatter, label='mt_frac')
    
    # 축과 제목 설정
    plt.xlabel('n_counts')
    plt.ylabel('n_genes')
    plt.title(f'AnnData {i} - n_counts vs n_genes with mt_frac (Normalized)')
    plt.grid(True)
    plt.show()

In [ ]:
Path10x_5_1='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_HHG_GEX/outs/filtered_feature_bc_matrix'
adata_pati_01 = sc.read_10x_mtx(Path10x_5_1,var_names='gene_symbols',cache=True)
Path10x_5_2='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KDI_GEX/outs/filtered_feature_bc_matrix'
adata_pati_02 = sc.read_10x_mtx(Path10x_5_2,var_names='gene_symbols',cache=True)
Path10x_5_3='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KGO_GEX/outs/filtered_feature_bc_matrix'
adata_pati_03 = sc.read_10x_mtx(Path10x_5_3,var_names='gene_symbols',cache=True)
Path10x_5_4='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KSS_GEX/outs/filtered_feature_bc_matrix'
adata_pati_04 = sc.read_10x_mtx(Path10x_5_4,var_names='gene_symbols',cache=True)
Path10x_5_5='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LAR_GEX/outs/filtered_feature_bc_matrix'
adata_pati_05 = sc.read_10x_mtx(Path10x_5_5,var_names='gene_symbols',cache=True)
Path10x_5_6='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LHS_GEX/outs/filtered_feature_bc_matrix'
adata_pati_06 = sc.read_10x_mtx(Path10x_5_6,var_names='gene_symbols',cache=True)
Path10x_5_7='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LJO_GEX/outs/filtered_feature_bc_matrix'
adata_pati_07 = sc.read_10x_mtx(Path10x_5_7,var_names='gene_symbols',cache=True)
Path10x_5_8='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_PHY_GEX/outs/filtered_feature_bc_matrix'
adata_pati_08 = sc.read_10x_mtx(Path10x_5_8,var_names='gene_symbols',cache=True)
Path10x_5_9='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/20240326/02_count/SC_5_1281_GEX/outs/filtered_feature_bc_matrix'
adata_pati_09 = sc.read_10x_mtx(Path10x_5_9,var_names='gene_symbols',cache=True)
Path10x_5_10='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/20240326/02_count/SC_5_8227_GEX/outs/filtered_feature_bc_matrix'
adata_pati_10 = sc.read_10x_mtx(Path10x_5_10,var_names='gene_symbols',cache=True)
Path10x_5_11='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/20240326/02_count/SC_5_9010_GEX/outs/filtered_feature_bc_matrix'
adata_pati_11 = sc.read_10x_mtx(Path10x_5_11,var_names='gene_symbols',cache=True)

adatap = [adata_pati_01,adata_pati_02,adata_pati_03,adata_pati_04,adata_pati_05\
           ,adata_pati_06,adata_pati_07,adata_pati_08,adata_pati_09,adata_pati_10,adata_pati_11]

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatap):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
for i, adata in enumerate(adatap):
    plt.figure(figsize=(8, 6))
    
    # 산점도 그리기 (c 파라미터에 mt_frac 사용, vmin/vmax 지정)
    scatter = plt.scatter(
        adata.obs['n_counts'], 
        adata.obs['n_genes'], 
        c=adata.obs['mt_frac'],  # 점의 색을 mt_frac으로 지정
        cmap='viridis',          # 컬러맵 선택
        alpha=0.7,
        vmin=0, vmax=1          
    )
    
    # 컬러바 추가
    plt.colorbar(scatter, label='mt_frac')
    
    # 축과 제목 설정
    plt.xlabel('n_counts')
    plt.ylabel('n_genes')
    plt.title(f'AnnData {i} - n_counts vs n_genes with mt_frac (Normalized)')
    plt.grid(True)
    plt.show()

In [ ]:
adata_normal = ad.concat(adatas, axis=0, label='sample', keys=['norm01', 'norm03', 'norm08','norm09','norm13','norm88','norm89','norm08ad','norm89ad'])
adata_patient = ad.concat(adatap, axis=0, label='sample', keys= ['pati01','pati02','pati03','pati04','pati05','pati06','pati07','pati08','pati09','pati10','pati11'])
# 병합 결과 확인
print(adata_normal.obs[['sample']])
print(adata_patient.obs[['sample']])

In [ ]:
sc.pl.violin(
    adata_normal,
    keys=["n_counts","n_genes"], 
    groupby="sample", 
    jitter=0.4, 
    rotation=90, 
    size=0.2, 
    scale="width", 
    stripplot=True, 
    show=False  # show=False로 설정해 Matplotlib에서 추가 작업 가능
)

# y축 범위 설정
plt.title("nCounts")
plt.ylim(0, 20000)  # 최대치를 30,000으로 설정
plt.show()

In [ ]:
Path10x_4_1='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-1_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_01 = sc.read_10x_mtx(Path10x_4_1,var_names='gene_symbols',cache=True)
Path10x_4_3='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-3_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_03 = sc.read_10x_mtx(Path10x_4_3,var_names='gene_symbols',cache=True)
Path10x_4_9='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-9_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_09 = sc.read_10x_mtx(Path10x_4_9,var_names='gene_symbols',cache=True)
Path10x_4_13='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-13_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_13 = sc.read_10x_mtx(Path10x_4_13,var_names='gene_symbols',cache=True)
Path10x_4_88='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/01_fastq_88/merged_4_88/4-88/outs/filtered_feature_bc_matrix/'
adata_norm_88 = sc.read_10x_mtx(Path10x_4_88,var_names='gene_symbols',cache=True)
Path10x_4_89='/home/jaehyunchoi/MASLD_collaboration/rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-89_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_89 = sc.read_10x_mtx(Path10x_4_89,var_names='gene_symbols',cache=True)
Path10x_5_1='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KDI_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_01 = sc.read_10x_mtx(Path10x_5_1,var_names='gene_symbols',cache=True)
Path10x_5_2='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KGO_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_02 = sc.read_10x_mtx(Path10x_5_2,var_names='gene_symbols',cache=True)
Path10x_5_3='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LAR_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_03 = sc.read_10x_mtx(Path10x_5_3,var_names='gene_symbols',cache=True)
Path10x_5_4='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LHS_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_04 = sc.read_10x_mtx(Path10x_5_4,var_names='gene_symbols',cache=True)
Path10x_5_5='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/20240326/02_count/SC_5_8227_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_05 = sc.read_10x_mtx(Path10x_5_5,var_names='gene_symbols',cache=True)
Path10x_5_6='/home/jaehyunchoi/MASLD_collaboration/rawdata/SCAID_liver/20240326/02_count/SC_5_9010_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_06 = sc.read_10x_mtx(Path10x_5_6,var_names='gene_symbols',cache=True)
adatas = [adata_norm_01,adata_norm_03,adata_norm_09,adata_norm_13,adata_norm_88,adata_norm_89]
adatap = [adata_pati_01,adata_pati_02,adata_pati_03,adata_pati_04,adata_pati_05,adata_pati_06]

In [ ]:
Path10x_4_1='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-1_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_01 = sc.read_10x_mtx(Path10x_4_1,var_names='gene_symbols',cache=True)
Path10x_4_3='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-3_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_03 = sc.read_10x_mtx(Path10x_4_3,var_names='gene_symbols',cache=True)
Path10x_4_9='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-9_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_09 = sc.read_10x_mtx(Path10x_4_9,var_names='gene_symbols',cache=True)
Path10x_4_13='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-13_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_13 = sc.read_10x_mtx(Path10x_4_13,var_names='gene_symbols',cache=True)
Path10x_4_88='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/01_fastq_88/merged_4_88/4-88/outs/filtered_feature_bc_matrix/'
adata_norm_88 = sc.read_10x_mtx(Path10x_4_88,var_names='gene_symbols',cache=True)
Path10x_4_89='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/241021_S010920240826-001_SNUH_SCAID_5_Liver/02_count/4-89_GEX/outs/filtered_feature_bc_matrix/'
adata_norm_89 = sc.read_10x_mtx(Path10x_4_89,var_names='gene_symbols',cache=True)
Path10x_5_1='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KDI_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_01 = sc.read_10x_mtx(Path10x_5_1,var_names='gene_symbols',cache=True)
Path10x_5_2='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_KGO_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_02 = sc.read_10x_mtx(Path10x_5_2,var_names='gene_symbols',cache=True)
Path10x_5_3='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LAR_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_03 = sc.read_10x_mtx(Path10x_5_3,var_names='gene_symbols',cache=True)
Path10x_5_4='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/SCAID_liver/240702_서울대병원_최광현_sc/02_count/SC_5_LHS_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_04 = sc.read_10x_mtx(Path10x_5_4,var_names='gene_symbols',cache=True)
Path10x_5_5='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/SCAID_liver/20240326/02_count/SC_5_8227_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_05 = sc.read_10x_mtx(Path10x_5_5,var_names='gene_symbols',cache=True)
Path10x_5_6='/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Rawdata/SCAID_liver/20240326/02_count/SC_5_9010_GEX/outs/filtered_feature_bc_matrix/'
adata_pati_06 = sc.read_10x_mtx(Path10x_5_6,var_names='gene_symbols',cache=True)
adatas = [adata_norm_01,adata_norm_03,adata_norm_09,adata_norm_13,adata_norm_88,adata_norm_89]
adatap = [adata_pati_01,adata_pati_02,adata_pati_03,adata_pati_04,adata_pati_05,adata_pati_06]

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatas):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
# 모든 AnnData 객체에 동일한 작업 적용
for i, adata in enumerate(adatap):
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    mt_genes = adata.var_names.str.startswith('MT-')
    ribo_genes = adata.var_names.str.contains('RPS|RPL')
    adata.obs['mt_counts'] = adata[:, mt_genes].X.sum(axis=1).A1
    adata.obs['ribo_counts'] = adata[:, ribo_genes].X.sum(axis=1).A1
    # 전체 유전자 발현량 대비 미토콘드리아 유전자 비율 계산
    adata.obs['mt_frac'] = adata.obs['mt_counts'] / adata.obs['n_counts']
    adata.obs['ribo_frac'] = adata.obs['ribo_counts'] / adata.obs['n_counts']
    print(f"AnnData {i}:")
    print(adata.obs[['n_counts', 'n_genes', 'mt_frac','ribo_frac']])

In [ ]:
for i, adata in enumerate(adatas):
    # 필터링 적용
    filtered_adata = adata[
        (adata.obs['n_genes'] > 250) & 
        (adata.obs['n_counts'] > 500) & 
        (adata.obs['n_counts'] < (np.mean(adata.obs['n_counts']) + 2 * np.std(adata.obs['n_counts']))) & 
        (adata.obs['mt_frac'] < 0.03),
        :
    ]
    print(filtered_adata.obs[['n_counts', 'n_genes', 'mt_frac']])
    # 필터링된 결과를 adatas 리스트에 다시 저장
    adatas[i] = filtered_adata

In [ ]:
for i, adata in enumerate(adatap):
    # 필터링 적용
    filtered_adata = adata[
        (adata.obs['n_genes'] > 250) & 
        (adata.obs['n_counts'] > 500) & 
        (adata.obs['n_counts'] < (np.mean(adata.obs['n_counts']) + 2 * np.std(adata.obs['n_counts']))) & 
        (adata.obs['mt_frac'] < 0.03),
        :
    ]
    print(filtered_adata.obs[['n_counts', 'n_genes', 'mt_frac']])
    # 필터링된 결과를 adatas 리스트에 다시 저장
    adatap[i] = filtered_adata

In [ ]:
adata_normal = ad.concat(
    adatas,
    label='sample',
    keys=['norm01','norm03','norm09','norm13','norm88','norm89'],
    index_unique='_'
)
adata_patient = ad.concat(
    adatap,
    label='sample',
    keys=['pati01','pati02','pati03','pati04','pati05','pati06'],
    index_unique='_'
)
print(adata_normal.obs[['sample']])
print(adata_patient.obs[['sample']])

In [ ]:
adata_merged = sc.concat(
    [adata_normal, adata_patient],
    keys=['normal', 'MASLD'],
    label='batch'
)

In [ ]:
adata_merged.obs

In [ ]:
adata_merged.obs_names[adata_merged.obs_names.duplicated()]

In [ ]:
adata_merged

In [ ]:
adata_merged.write("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_raw.h5ad")